# Financial Analysis: Historical Fundamentals and Price Performance

This notebook begins the financial analysis using the saved cleaned annual figures and daily closing prices in `data/cleaned`. Generate them by running Section 12 of `data_exploration.ipynb`. Outlier flags are prompts for review, not exclusions. Results are descriptive and do not establish investment value or forecast future returns.

Run from the project root or this notebook’s directory.

## 1. Load and prepare the cleaned data

Load the approved cleaned CSV files. Annual figures have an `available_date` indicating when each observation became available. A fiscal-year value must not be used before that date in any later backtest. Closing-price returns exclude dividends, and the benchmark is a price index rather than a total-return series.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_cleaned_data

annual, prices = load_cleaned_data(project_root / "data" / "cleaned")
print(f"Annual rows: {len(annual):,}; daily rows: {len(prices):,}")

## 2. Operating performance and growth

Start with sales, profit, operating cash flow, and free cash flow. Profit margin is profit divided by sales; FCF margin is free cash flow divided by sales. Year-over-year growth is shown only when the prior-year value is non-zero and the arithmetic is finite. Negative values and sign changes can make growth percentages misleading, so inspect the underlying amounts alongside them. These measures support within-company trend review; comparisons across sectors require care. The charts show sales for all companies, then split profit-margin history into two groups using the latest-year median: the top 50% at or above the median and the lower 50% below it.

In [ ]:
annual = annual.sort_values(["ticker", "year"]).copy()
annual["profit_margin"] = annual["profit_gbp_m"] / annual["sales_gbp_m"].where(annual["sales_gbp_m"].ne(0))
annual["fcf_margin"] = annual["free_cash_flow_gbp_m"] / annual["sales_gbp_m"].where(annual["sales_gbp_m"].ne(0))
for metric in ["sales_gbp_m", "profit_gbp_m", "operating_cash_flow_gbp_m", "free_cash_flow_gbp_m"]:
    previous = annual.groupby("ticker")[metric].shift()
    annual[f"{metric}_yoy"] = (annual[metric] / previous.where(previous.ne(0))) - 1
latest_year = annual["year"].max()
latest = annual.loc[annual["year"].eq(latest_year)].copy()
summary_cols = ["ticker", "company", "sector", "year", "sales_gbp_m", "profit_gbp_m", "profit_margin", "operating_cash_flow_gbp_m", "free_cash_flow_gbp_m", "fcf_margin", "debt_gbp_m", "cash_gbp_m"]
display(latest[summary_cols].sort_values("free_cash_flow_gbp_m", ascending=False, na_position="last").round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
# Order the lines by latest-year free cash flow, highest to lowest.
ticker_order = latest.sort_values("free_cash_flow_gbp_m", ascending=False, na_position="last")["ticker"]
profit_margin_median = latest["profit_margin"].median()
top_margin_tickers = set(latest.loc[latest["profit_margin"].ge(profit_margin_median), "ticker"])
lower_margin_tickers = set(latest.loc[latest["profit_margin"].lt(profit_margin_median), "ticker"])
for ticker in ticker_order:
    group = annual.loc[annual["ticker"].eq(ticker)]
    axes[0].plot(group["year"], group["sales_gbp_m"], alpha=0.65, linewidth=1, label=ticker)
    if ticker in top_margin_tickers:
        axes[1].plot(group["year"], group["profit_margin"], alpha=0.65, linewidth=1, label=ticker)
    elif ticker in lower_margin_tickers:
        axes[2].plot(group["year"], group["profit_margin"], alpha=0.65, linewidth=1, label=ticker)
axes[0].set(title="Sales by company", xlabel="Fiscal year", ylabel="GBP millions")
axes[1].set(title="Top 50% profit margins", xlabel="Fiscal year", ylabel="Profit / sales")
axes[2].set(title="Lower 50% profit margins", xlabel="Fiscal year", ylabel="Profit / sales")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(title="Ticker (2025 FCF order)", bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2)
fig.tight_layout()
plt.show()

### Latest-year free cash flow ranking

This bar chart orders companies by 2025 free cash flow, from highest to lowest. Companies with missing FCF remain excluded from the ranking because there is no value to plot.


In [ ]:
fcf_ranking = latest.dropna(subset=["free_cash_flow_gbp_m"]).sort_values("free_cash_flow_gbp_m", ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(fcf_ranking["ticker"], fcf_ranking["free_cash_flow_gbp_m"])
ax.set(title=f"{latest_year} free cash flow by company", xlabel="Ticker", ylabel="Free cash flow (GBP millions)")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## 3. Balance sheet and cash generation

Debt-to-cash is a simple liquidity context measure, not a substitute for net debt, maturity, or interest coverage analysis. It is undefined where cash is zero and can be unusually large when cash is small. FCF conversion is free cash flow divided by operating cash flow; it is not meaningful when operating cash flow is zero or close to zero. Negative results and ratios above 100% can occur, so review components rather than treating thresholds as pass/fail.

In [ ]:
latest["debt_to_cash"] = latest["debt_gbp_m"] / latest["cash_gbp_m"].where(latest["cash_gbp_m"].ne(0))
latest["fcf_to_operating_cash_flow"] = latest["free_cash_flow_gbp_m"] / latest["operating_cash_flow_gbp_m"].where(latest["operating_cash_flow_gbp_m"].ne(0))
display(latest[["ticker", "sector", "debt_gbp_m", "cash_gbp_m", "debt_to_cash", "operating_cash_flow_gbp_m", "free_cash_flow_gbp_m", "fcf_to_operating_cash_flow"]].sort_values(["sector", "ticker"]).round(3))

## 4. Price performance and risk context

Calculate daily price-only returns from cleaned closing prices. The summary shows total price change, annualized volatility from daily returns, and maximum drawdown over the available history. The first and last available observations differ by ticker only if there are missing values; inspect date coverage before comparing. The benchmark series is included as context. These figures omit dividends, transaction costs, and taxes, and should not be called total shareholder returns.

In [ ]:
price_data = prices.set_index("Date").sort_index()
returns = price_data.pct_change(fill_method=None)
trading_days = 252
records = []
for ticker in price_data.columns:
    series = price_data[ticker].dropna()
    r = returns[ticker].dropna()
    wealth = (1 + r).cumprod()
    drawdown = wealth / wealth.cummax() - 1
    elapsed_years = (series.index[-1] - series.index[0]).days / 365.25
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1 / elapsed_years) - 1 if elapsed_years > 0 and series.iloc[-1] > 0 and series.iloc[0] > 0 else float("nan")
    records.append({"ticker": ticker, "start": series.index[0], "end": series.index[-1],
                    "observations": len(series), "price_change": series.iloc[-1] / series.iloc[0] - 1,
                    "annualized_price_growth": cagr,
                    "annualized_volatility": r.std() * trading_days ** 0.5,
                    "maximum_drawdown": drawdown.min()})
price_summary = pd.DataFrame(records).set_index("ticker")
display(price_summary.sort_values("annualized_price_growth", ascending=False).round(4))

In [ ]:
# Rebase all series to 100 on their first observed date for a visual comparison.
rebased = price_data.divide(price_data.iloc[0]).mul(100)
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in rebased.columns:
    ax.plot(rebased.index, rebased[ticker], linewidth=1, alpha=0.7, label=ticker)
ax.set(title="Rebased closing prices (price-only, start = 100)", xlabel="Date", ylabel="Index")
ax.grid(alpha=0.25)
ax.legend(title="Ticker", bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2)
fig.tight_layout()
plt.show()

## 5. Timing and interpretation

The annual figures span fiscal years 2016–2025, and the daily price history spans 2016–2025. This notebook reports historical descriptions; it does not align annual fundamentals to returns or test a strategy. For any future valuation or backtest, join fundamentals using `available_date` (not just fiscal year) to prevent look-ahead bias. The supplied prices are closing prices and do not account for dividends, corporate actions beyond the known MGS correction, trading costs, or taxes. Confirm definitions and source units before using ratios in decisions.

In [ ]:
print("Annual fiscal-year coverage:", annual["year"].min(), "to", annual["year"].max())
print("Annual availability-date coverage:", annual["available_date"].min(), "to", annual["available_date"].max())
print("Daily price coverage:", prices["Date"].min(), "to", prices["Date"].max())
print("Missing annual values by field:")
display(annual.isna().sum().rename("missing_count").to_frame())